# L.I.V.E. — síntese de voz no Colab

Gera falas de números com motores de licença comercial clara. **Só dados sintéticos ou públicos aqui** (Princípio IV): a gravação real do PO nunca vem para a nuvem.

Kokoro roda em CPU; Chatterbox e Parler pedem GPU (Ambiente de execução → Alterar tipo → GPU).

In [ ]:
!git clone --depth 1 -b 001-gramatica-dataprep https://github.com/yuriwinchest/live-engine-training.git
%cd live-engine-training/training
# Sem requirements.txt aqui: as versões fixadas brigam com o numpy/numba do Colab.
!pip install -q -e .

## Motor
Instale **um** motor por sessão (eles fixam versões diferentes de torch).

In [ ]:
ENGINE = "kokoro"  # kokoro | chatterbox | parler
if ENGINE == "kokoro":
    !apt-get -qq -y install espeak-ng > /dev/null
    !pip install -q kokoro
elif ENGINE == "chatterbox":
    !pip install -q chatterbox-tts
else:
    !pip install -q git+https://github.com/huggingface/parler-tts.git

Chatterbox clona a voz de clipes de referência **CC0** (Common Voice pt). Envie a pasta de referência para `/content/refs` e use `--engine-option references=/content/refs`.

In [ ]:
COUNT, SEED = 3000, 7
OPTS = "" if ENGINE != "chatterbox" else "--engine-option references=/content/refs"
!live-lab synth plan --engine {ENGINE} {OPTS} --out data/jobs-{ENGINE}.jsonl --count {COUNT} --seed {SEED}
!live-lab synth render --engine {ENGINE} {OPTS} --jobs data/jobs-{ENGINE}.jsonl --out data/clean/{ENGINE}

Se a sessão cair, rode a célula de novo: a renderização continua de onde parou.

## Levar o resultado
Compacte e baixe (ou copie para o seu Drive). A degradação com ruído (`live-lab prep`) pode rodar aqui ou localmente.

In [ ]:
!cd data/clean && zip -qr /content/clean-{ENGINE}.zip {ENGINE}
from google.colab import files
files.download(f"/content/clean-{ENGINE}.zip")